In [7]:
# =============================================================================
# Ablation Study: Tensors to Detailed Results (v3 - Dummy Ready)
# =============================================================================
import os
import json
import torch
import numpy as np
import pandas as pd
import re
from typing import Dict, List, Tuple, Any
from sklearn.preprocessing import StandardScaler

# Mount Google Drive
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# =============================================================================
# 1. CONFIGURATION & FEATURE DISCOVERY
# =============================================================================

TENSOR_ROOT = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/tensors"
TIME_SERIES_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/paper_gdrive/by-week-keyword/sem_dtw_geo_enriched_dummies.parquet"
GRAPH_FOLDER_PATH = "/content/drive/MyDrive/Colab Notebooks/master_thesis_gdrive/sebs_keyword_graph_knn"

TARGET_COL = 'cpc_week'
TEST_WEEKS_LAST = 12
VAL_RATIO = 0.25
NUM_NODES = 1811

# --- Dynamic Feature Discovery ---
_temp_df = pd.read_parquet(TIME_SERIES_CSV_PATH, columns=None)
all_cols = _temp_df.columns.tolist()

# Define Groups based on columns actually present in the file
CORE_FEAT = ['impressions_sum', 'cpc_week']
GEO_CONTINENT_FEAT = [c for c in all_cols if c.startswith('detected_continent_')]
GEO_COUNTRY_FEAT = [c for c in all_cols if c.startswith('detected_country_')]
GEO_CITY_FEAT = [c for c in all_cols if c.startswith('detected_city_')]
SEARCH_FEAT = ['n_st_branded_search', 'n_st_generic_search']

# Fallback for Semantic PC (using neighbors since PC is missing in new parquet)
SEM_PC_FEAT = [c for c in all_cols if c.startswith('semantic_neighbour_')][:4]
if not SEM_PC_FEAT:
    SEM_PC_FEAT = ['sem_pc_0', 'sem_pc_1', 'sem_pc_2', 'sem_pc_4'] # Keep original if found

EXPERIMENTS = {
    "core_continent": CORE_FEAT + GEO_CONTINENT_FEAT,
    "core_country": CORE_FEAT + GEO_COUNTRY_FEAT,
    "core_city": CORE_FEAT + GEO_CITY_FEAT,
    "core_continent_search": CORE_FEAT + GEO_CONTINENT_FEAT + SEARCH_FEAT,
    "core_country_search": CORE_FEAT + GEO_COUNTRY_FEAT + SEARCH_FEAT,
    "core_continent_sempc": CORE_FEAT + GEO_CONTINENT_FEAT + SEM_PC_FEAT,
}

# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def get_scaler_for_experiment(exp_name: str, time_series_path: str):
    features = EXPERIMENTS[exp_name]

    # Check for missing columns before loading
    missing = [c for c in features if c not in all_cols]
    if missing:
        raise ValueError(f"Experiment '{exp_name}' has missing columns in Parquet: {missing}")

    df = pd.read_parquet(time_series_path, columns=features + ['week'])

    # Parse Weeks
    if df['week'].dtype == object:
        parts = df['week'].astype(str).str.split('-', expand=True)
        df['week'] = pd.to_numeric(parts[1]) * 100 + pd.to_numeric(parts[0])
    else:
        df['week'] = pd.to_numeric(df['week'])

    # Split (Matching Training Logic)
    weeks = np.array(sorted(df['week'].unique()))
    trainval_weeks = weeks[:-TEST_WEEKS_LAST]
    split_idx = int(len(trainval_weeks) * (1 - VAL_RATIO))
    train_weeks = trainval_weeks[:split_idx]

    df_train = df[df['week'].isin(train_weeks)].copy()
    target_idx = features.index(TARGET_COL)

    # Force float32 to avoid 'no callable log1p' error
    X_flat = df_train[features].values.astype(np.float32)
    X_flat[:, target_idx] = np.log1p(X_flat[:, target_idx])

    scaler = StandardScaler()
    scaler.fit(X_flat)
    return scaler, target_idx

def calculate_advanced_metrics(preds, targets, scaler, target_col_idx):
    if isinstance(preds, torch.Tensor): preds = preds.cpu().numpy()
    if isinstance(targets, torch.Tensor): targets = targets.cpu().numpy()

    # Flatten for inverse transform
    p_flat = preds.reshape(-1, 1)
    t_flat = targets.reshape(-1, 1)

    num_features = scaler.mean_.shape[0]
    d_p = np.zeros((len(p_flat), num_features))
    d_t = np.zeros((len(t_flat), num_features))

    d_p[:, target_col_idx] = p_flat[:, 0]
    d_t[:, target_col_idx] = t_flat[:, 0]

    # Inverse transform and Reverse Log1p
    real_preds = np.expm1(scaler.inverse_transform(d_p)[:, target_col_idx])
    real_targets = np.expm1(scaler.inverse_transform(d_t)[:, target_col_idx])
    real_preds = np.maximum(real_preds, 0.0)

    try:
        P = real_preds.reshape(-1, NUM_NODES)
        A = real_targets.reshape(-1, NUM_NODES)

        node_rmse = np.sqrt(np.mean((P - A) ** 2, axis=0))
        num = np.abs(P - A)
        den = (np.abs(P) + np.abs(A)) / 2.0
        node_smape = 100 * np.mean(num / (den + 1e-8), axis=0)

        return {
            'avg_smape': np.mean(node_smape), 'med_smape': np.median(node_smape), 'std_smape': np.std(node_smape),
            'avg_rmse': np.mean(node_rmse), 'med_rmse': np.median(node_rmse), 'std_rmse': np.std(node_rmse)
        }
    except Exception as e:
        return None

# =============================================================================
# 3. MAIN EXECUTION
# =============================================================================

results_list = []
for exp_name in sorted(EXPERIMENTS.keys()):
    exp_dir = os.path.join(TENSOR_ROOT, f"exp_{exp_name}")
    if not os.path.exists(exp_dir): continue

    print(f"\n>>> Processing: {exp_name}")
    try:
        scaler, target_idx = get_scaler_for_experiment(exp_name, TIME_SERIES_CSV_PATH)
    except Exception as e:
        print(f"  FAILED: {e}")
        continue

    all_files = os.listdir(exp_dir)
    pred_files = [f for f in all_files if f.endswith('_predictions.pt')]

    for p_file in sorted(pred_files):
        t_file = p_file.replace('_predictions.pt', '_targets.pt')
        if t_file not in all_files: continue

        # Extract model and horizon: {Model}_{Exp}_h{H}_predictions.pt
        match = re.search(r'(.+)_' + re.escape(exp_name) + r'_h(\d+)_predictions\.pt', p_file)
        model_name, horizon = (match.group(1), int(match.group(2))) if match else (p_file, "??")

        print(f"    - {model_name} (H={horizon})")
        preds = torch.load(os.path.join(exp_dir, p_file), map_location='cpu')
        targets = torch.load(os.path.join(exp_dir, t_file), map_location='cpu')

        metrics = calculate_advanced_metrics(preds, targets, scaler, target_idx)
        if metrics:
            results_list.append({'Experiment': exp_name, 'Horizon': horizon, 'Model': model_name, **metrics})

if results_list:
    df_final = pd.DataFrame(results_list).sort_values(['Experiment', 'Horizon', 'avg_smape'])
    df_final.to_csv("ablation_detailed_results_v3.csv", index=False)
    print("\n" + "="*50 + "\nSUCCESS: Saved to ablation_detailed_results_v3.csv\n" + "="*50)
    print(df_final.head(10))


>>> Processing: core_city
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (H=6)
    - GConvLSTM (H=12)
    - GConvLSTM (H=1)
    - GConvLSTM (H=6)
    - GraphWaveNet (H=12)
    - GraphWaveNet (H=1)
    - GraphWaveNet (H=6)

>>> Processing: core_continent
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (H=6)
    - GConvLSTM (H=12)
    - GConvLSTM (H=1)
    - GConvLSTM (H=6)
    - GraphWaveNet (H=12)
    - GraphWaveNet (H=1)
    - GraphWaveNet (H=6)

>>> Processing: core_continent_search
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (H=6)
    - GConvLSTM (H=12)
    - GConvLSTM (H=1)
    - GConvLSTM (H=6)
    - GraphWaveNet (H=12)
    - GraphWaveNet (H=1)
    - GraphWaveNet (H=6)

>>> Processing: core_continent_sempc
    - AGCRN (H=12)
    - AGCRN (H=1)
    - AGCRN (H=6)
    - DCRNN (H=12)
    - DCRNN (H=1)
    - DCRNN (